In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q08/annotated/checkpoints/pre_cell_7.pickle

trying: ['lineitem']


me:  2
trying: ['load_supplier']
me:  3
trying: ['orders']


me:  4
trying: ['DATA_ROOT']
me:  0
trying: ['nation']
me:  6
trying: ['load_customer']
me:  5
trying: ['part']
me:  1
trying: ['load_orders']
me:  4
trying: ['supplier']
me:  3
trying: ['load_part']
me:  1
trying: ['customer']
me:  5
trying: ['tpch_parent']
me:  0
trying: ['load_nation']
me:  6
trying: ['load_region']
me:  7
trying: ['region']
me:  7
trying: ['load_lineitem']
me:  2
trying: ['STORAGE_OPTS']
me:  0
trying: ['pd']
me:  0


Declaring variable DATA_ROOT
Declaring variable tpch_parent
Declaring variable STORAGE_OPTS
Declaring variable pd
Declaring variable part
Declaring variable load_part
Declaring variable lineitem
Declaring variable load_lineitem
Declaring variable load_supplier
Declaring variable supplier
Declaring variable orders
Declaring variable load_orders
Declaring variable load_customer
Declaring variable customer
Declaring variable nation
Declaring variable load_nation
Declaring variable load_region
Declaring variable region


In [4]:
%%RecordEvent
%%time
### cell 7 ###

# Step 0: filter region to AMERICA (needed by Step 1)
region_filtered = region[
    region.R_NAME == "AMERICA"
][["R_REGIONKEY"]]

# Step 1: get nation keys for America region without an extra merge
america_nations = nation[
    nation.N_REGIONKEY.isin(region_filtered.R_REGIONKEY)
][["N_NATIONKEY"]]

# Step 2: filter customers in America region via a GPU-friendly isin
cust_in_america = customer[
    customer.C_NATIONKEY.isin(america_nations.N_NATIONKEY)
][["C_CUSTKEY"]]

# Step 3: filter orders by date and customer region, add O_YEAR
orders_filtered = (
    orders[
        (orders.O_ORDERDATE >= "1995-01-01")
        & (orders.O_ORDERDATE <  "1997-01-01")
        & orders.O_CUSTKEY.isin(cust_in_america.C_CUSTKEY)
    ]
    .assign(O_YEAR=lambda df: df.O_ORDERDATE.dt.year)
    [["O_ORDERKEY", "O_YEAR"]]
)

# Step 4: bring in supplier nation names in one merge
supplier_nations = (
    supplier[["S_SUPPKEY", "S_NATIONKEY"]]
    .merge(
        nation[["N_NATIONKEY", "N_NAME"]]
              .rename(columns={"N_NAME": "NATION"}),
        left_on="S_NATIONKEY", right_on="N_NATIONKEY"
    )[["S_SUPPKEY", "NATION"]]
)

# Step 5: prefilter parts and compute lineitem volume
part_filtered = part[
    part.P_TYPE == "ECONOMY ANODIZED STEEL"
][["P_PARTKEY"]]

lineitem_filtered = lineitem.assign(
    VOLUME=lineitem.L_EXTENDEDPRICE * (1.0 - lineitem.L_DISCOUNT)
)[["L_PARTKEY", "L_SUPPKEY", "L_ORDERKEY", "VOLUME"]]

# Step 6: chain the remaining joins (3 merges instead of 7)
total = (
    lineitem_filtered
    .merge(orders_filtered,  left_on="L_ORDERKEY", right_on="O_ORDERKEY")
    .merge(part_filtered,    left_on="L_PARTKEY",   right_on="P_PARTKEY")
    .merge(supplier_nations, left_on="L_SUPPKEY",   right_on="S_SUPPKEY")
    [["VOLUME", "O_YEAR", "NATION"]]
)

# Step 7: compute Brazil volume and market share on GPU end-to-end
total = (
    total
    .assign(BRAZIL_VOL=total.VOLUME.where(total.NATION == "BRAZIL", 0))
    .groupby("O_YEAR")[["VOLUME", "BRAZIL_VOL"]]
    .sum()
    .reset_index()
    .assign(MKT_SHARE=lambda df: df.BRAZIL_VOL / df.VOLUME)[["O_YEAR", "MKT_SHARE"]]
    .sort_values("O_YEAR")
)


CPU times: user 130 ms, sys: 40.8 ms, total: 171 ms
Wall time: 193 ms


In [5]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q08/rewritten/o4_mini_high/checkpoints/post_cell_7_try_2.pickle

migration speed (bps): 813529237.4190104
---------------------------
variables to migrate:
load_lineitem 144
load_orders 144
part 48
orders 48
load_part 144
america_nations 48
lineitem_filtered 48
load_customer 144
orders_filtered 48
customer 48
cust_in_america 48
total 48
region_filtered 48
part_filtered 48
supplier_nations 48
tpch_parent 54
STORAGE_OPTS 64
load_nation 144
nation 48
region 48
load_region 144
DATA_ROOT 80
pd 72
load_supplier 144
supplier 48
lineitem 48
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/tpch/notebooks/q08/rewritten/o4_mini_high/checkpoints/post_cell_7_try_2.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables []
Intermediate variables ['part']
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables ['lineitem']
Intermediate variables []
Future variables ['part']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables ['supplier']
Future variables ['lineitem', 'part']
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables []
Intermediate variables ['orders']
Future variables ['lineitem', 'supplier', 'part']
Modified dataframes
======= Cell 4 =======
Input variables []
Active variables ['customer']
Intermediate variables []
Future variables ['lineitem', 'supplier', 'orders', 'part']
Modified dataframes
======= Cell 5 =======
Input variables []
Active variables []
Intermediate variables ['nation']
Future variables ['orders', 'customer', 'part', 'supplier', 'lineitem']
Modified dataframes
======= Cell 6 =======
Input varia

In [7]:

with open("/home/dias-benchmarks/tpch/notebooks/q08/opt_cell_exec_info_7_try_2.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[7], f)


In [ ]:
opt_output = Out.get(4)

In [ ]:
total_opt = total
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q08/annotated/checkpoints/post_cell_7.pickle

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output
